# 07 · Fetch the citer pool and the candidate corpus (Phase 5, P1–P3)

Phase 5 replaces the fixed candidate pools with a corpus of about 10k papers and adds a co-citation graph. Design and decisions: `research/audit/PHASE5_DESIGN.md`.

- **P1.** Refetch the 1,733 v2 works with `doi` and `ids` (needed for deduplication) into a new cache. `data/v2/openalex_works.jsonl` stays unchanged.
- **P2. Citer pool.** For each query, an OpenAlex title search for works published **before** the query with at least 10 references (first 100 results). Selection reads only the query's title, never its references.
- **P3. Candidate corpus.** All eligible positives, plus every eligible work cited by at least *m* pool citers. Query records never count as citers. *m* is chosen from corpus size alone, before any method runs.

All raw responses are cached under `data/v2/corpus/`, so a rerun makes no API calls. The OpenAlex key is read from `OPENALEX_API_KEY` and never written to disk.

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")

RESEARCH = Path.cwd().parent if Path.cwd().name == "v2" else Path.cwd()
sys.path.insert(0, str(RESEARCH))

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [2]:
TARGET_ELIGIBLE = 10_000   # corpus size target (decision 2)
P0_ELIGIBLE_RATE = 63 / 100  # P0: of 100 sampled pool references, 63 came back with title + abstract

In [3]:
import json

from src.pipeline_v2 import load_eval_sets, load_v1_queries

ds = load_eval_sets("random")  # queries, positives and splits are identical in both benchmarks
sets, splits = ds["sets"], ds["splits"]
query_ids = {q["id"] for q in load_v1_queries()}  # all 174 v1 query records: never citers, never candidates
positives = sorted({p for s in sets for p in s["positives"]})
print(f"{len(sets)} queries ({sum(v == 'dev' for v in splits.values())} dev / {sum(v == 'test' for v in splits.values())} test), "
      f"{len(positives)} distinct positives, {len(query_ids)} query records excluded as citers")

168 queries (50 dev / 118 test), 1180 distinct positives, 174 query records excluded as citers


### P1 · Refetch the v2 works with `doi` and `ids`

In [4]:
from src.corpus import fetch_works

old_ids = [json.loads(line)["id"] for line in (RESEARCH / "data" / "v2" / "openalex_works.jsonl").open()]
works, not_found = fetch_works(old_ids)
print(f"{len(works)}/{len(old_ids)} returned; not returned now: {len(not_found)}")
print("not-returned positives:", sorted(set(positives) & not_found))
print("with DOI:", sum(bool(w.get("doi")) for w in works.values()))

1726/1733 returned; not returned now: 7
not-returned positives: ['https://openalex.org/W2951482518', 'https://openalex.org/W3090325631', 'https://openalex.org/W3168183489', 'https://openalex.org/W4226369848']
with DOI: 1689


### P2 · Citer pool: one title search per query

In [5]:
from tqdm.auto import tqdm

from src.corpus import citer_pool, fetch_citers

searches = [fetch_citers(s["pi_id"], s["pi_title"], s["pi_date"]) for s in tqdm(sets, desc="citer searches")]
pool = citer_pool(searches)

per_query = pd.DataFrame([{"pi_id": s["pi_id"], "split": splits[s["pi_id"]], "matches": s["count"],
                           "returned": len(s["results"])} for s in searches])
print(per_query[["matches", "returned"]].describe().round(1).T)
print(f"queries with < 100 earlier matches: {(per_query.returned < 100).sum()}")
print(f"citer pool: {len(pool)} distinct works from {per_query.returned.sum()} search results")
print("fetched on:", sorted({s["fetched"] for s in searches}))

          count    mean      std  min   25%    50%     75%       max
matches   168.0  7911.9  42721.2  0.0  23.8  195.0  1348.0  484181.0
returned  168.0    70.2     40.3  0.0  23.8  100.0   100.0     100.0
queries with < 100 earlier matches: 72
citer pool: 10663 distinct works from 11791 search results
fetched on: ['2026-10-12']


In [6]:
import re

def norm_title(t):
    return re.sub(r"\W+", " ", (t or "").lower()).strip()

query_titles = {norm_title(s["pi_title"]): s["pi_id"] for s in sets}
in_pool_queries = sorted(set(pool) & query_ids)
same_title = sorted((cid, query_titles[norm_title(w["title"])]) for cid, w in pool.items()
                    if norm_title(w["title"]) in query_titles and cid not in query_ids)
years = pd.Series([w["publication_date"][:4] for w in pool.values() if w.get("publication_date")]).astype(int)
print(f"query records in the pool: {len(in_pool_queries)}")
print(f"pool works with a query's exact title (other ids): {len(same_title)}")
for cid, qid in same_title:
    print("  ", cid, "~", qid, "|", pool[cid]["publication_date"], pool[cid]["title"][:80])
print("pool publication years:", years.describe()[["min", "25%", "50%", "75%", "max"]].to_dict())

query records in the pool: 2
pool works with a query's exact title (other ids): 13
   https://openalex.org/W2044826551 ~ https://openalex.org/W4242865819 | 2013-09-09 Electronic cigarettes for smoking cessation
   https://openalex.org/W2076719248 ~ https://openalex.org/W4242865819 | 2014-10-10 Electronic Cigarettes for Smoking Cessation
   https://openalex.org/W2482703967 ~ https://openalex.org/W4242865819 | 2016-08-04 Electronic Cigarettes for Smoking Cessation
   https://openalex.org/W2747717603 ~ https://openalex.org/W4242865819 | 2016-09-13 Electronic cigarettes for smoking cessation
   https://openalex.org/W2785150556 ~ https://openalex.org/W4242865819 | 2018-01-17 Electronic cigarettes for smoking cessation
   https://openalex.org/W3212080866 ~ https://openalex.org/W3212282753 | 2021-11-06 STROCSS 2021: Strengthening the reporting of cohort, cross-sectional and case-co
   https://openalex.org/W4210741472 ~ https://openalex.org/W4242865819 | 2012-11-26 Electronic cigarettes for sm

### Coverage: positives the graph can reach

A positive can be recovered through co-citation only if some allowed citer cites it: a pool work that is not a query record and is published before the query. Same-title copies of queries are dropped in notebook 08; here only query records are excluded.

In [7]:
from src.corpus import positive_coverage

cov = pd.DataFrame(positive_coverage(sets, pool, exclude=query_ids))
cov["split"] = cov.pi_id.map(splits)
cov["share"] = cov.n_covered / cov.n_pos
summary = cov.groupby("split").agg(queries=("pi_id", "size"), positive_slots=("n_pos", "sum"),
                                   covered=("n_covered", "sum"), mean_share=("share", "mean"),
                                   no_coverage=("n_covered", lambda x: int((x == 0).sum())))
summary["slot_share"] = summary.covered / summary.positive_slots
summary.round(3)

,queries,positive_slots,covered,mean_share,no_coverage,slot_share
split,,,,,,
dev,50,383,237,0.609,4,0.619
test,118,864,538,0.614,2,0.623


### Corpus size by *m*

In [8]:
from src.corpus import ref_counts

counts = ref_counts(pool, exclude=query_ids)
pos_set = set(positives)
rows = []
for m in range(1, 11):
    ids_m = {w for w, c in counts.items() if c >= m} - query_ids
    rows.append({"m": m, "works": len(ids_m), "positives_among_them": len(ids_m & pos_set),
                 "est_eligible_non_positive": round(len(ids_m - pos_set) * P0_ELIGIBLE_RATE)})
by_m = pd.DataFrame(rows)
by_m["est_corpus"] = by_m.est_eligible_non_positive + len(positives)
M = int(by_m.loc[(by_m.est_corpus - TARGET_ELIGIBLE).abs().idxmin(), "m"])
print(f"chosen m = {M} (closest estimated corpus size to {TARGET_ELIGIBLE:,})")
by_m

chosen m = 5 (closest estimated corpus size to 10,000)


,m,works,positives_among_them,est_eligible_non_positive,est_corpus
0,1,533466,755,335608,336788
1,2,79651,565,49824,51004
2,3,32776,454,20363,21543
3,4,18840,389,11624,12804
4,5,12895,343,7908,9088
5,6,9507,309,5795,6975
6,7,7361,284,4459,5639
7,8,5971,265,3595,4775
8,9,5028,244,3014,4194
9,10,4283,227,2555,3735


### P3 · Fetch the corpus and keep eligible works

In [9]:
from src.eval_v2 import eligible
from src.openalex import normalize_work

candidates = ({w for w, c in counts.items() if c >= M} - query_ids) | pos_set
corpus_raw, corpus_missing = fetch_works(candidates)
norm = {pid: normalize_work(w) for pid, w in corpus_raw.items()}
corpus_ids = sorted(pid for pid, rec in norm.items() if eligible(rec) or pid in pos_set)
non_pos = [pid for pid in corpus_ids if pid not in pos_set]
print(f"requested {len(candidates)}, returned {len(corpus_raw)}, not returned {len(corpus_missing)}")
print(f"eligible non-positives: {len(non_pos)} of {len(candidates - pos_set) - len(corpus_missing - pos_set)} returned "
      f"({len(non_pos) / max(1, len(candidates - pos_set) - len(corpus_missing - pos_set)):.1%})")
print(f"positives returned: {len(pos_set & set(corpus_raw))}/{len(pos_set)}; "
      f"positives now ineligible: {sum(not eligible(norm[p]) for p in pos_set if p in norm)}")
print(f"corpus: {len(corpus_ids)} works ({len(pos_set & set(corpus_ids))} positives)")

requested 13732, returned 13078, not returned 654
eligible non-positives: 8530 of 11902 returned (71.7%)
positives returned: 1176/1180; positives now ineligible: 1
corpus: 9706 works (1176 positives)


In [10]:
manifest = {
    "m": M, "target_eligible": TARGET_ELIGIBLE, "citers_per_query": 100,
    "search_dates": sorted({s["fetched"] for s in searches}),
    "n_citers": len(pool), "n_corpus": len(corpus_ids),
    "n_positives_in_corpus": len(pos_set & set(corpus_ids)),
    "corpus_ids": corpus_ids,
}
out = RESEARCH / "v2" / "corpus" / "corpus_manifest.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(manifest, indent=0))
print(out.relative_to(RESEARCH), f"{out.stat().st_size / 1e3:.0f} kB")

v2/corpus/corpus_manifest.json 349 kB


### Findings

- **Cost.** 168 searches ($0.168) plus about 160 list calls. One run with a free API key, all responses cached (search date 2026-10-12).
- **P1.** 1,726 of the 1,733 v2 works are still in OpenAlex, 1,689 with a DOI. The 7 that are gone include 4 positives.
- **Citer pool.** The title searches returned 11,791 results, 10,663 distinct works (median publication year 2020).
  - Earlier matches per query range from 0 to 484,181 (median 195).
  - 72 queries have fewer than 100 earlier matches and 5 have none. The search behaves like an AND over all title words, so long titles match little.
- **Leakage risk is real.** 2 query records and 13 same-title copies of queries came back as pool works. Notebook 08 removes them (rule R3).
  - Five are earlier versions of the Cochrane review *Electronic cigarettes for smoking cessation* (2012–2018), whose reference lists overlap the query's.
  - Others are working-paper versions of *The Evolution of Work from Home* and copies of STROCSS 2021.
- **Coverage.** 62% of positive slots (dev 61.9%, test 62.3%) are cited by at least one pool citer that is not a query record and predates the query. Only 4 dev and 2 test queries have no covered positive. Same-title copies are still counted here.
- **Corpus.** m = 5 gives the estimate closest to 10k: 12,895 works are cited by at least 5 pool citers, 343 of them positives.
  - Of the non-positives fetched, 71.7% have a title and an abstract (P0 estimate: 63%).
  - **The corpus holds 9,706 works: 1,176 positives and 8,530 non-positives.** 4 positives are no longer in OpenAlex and 1 positive lost its abstract (kept as a positive).
  - Ids are in `v2/corpus/corpus_manifest.json`.
